# 06 — Segmentation Cross-Experiment Comparison

Loads `cv_summary.csv`, `cv_class_summary.csv`, and `cv_results.csv` for **all**
completed segmentation experiments under a given dataset + split_scheme, and produces:

- **Ranked comparison table** — all experiments sorted by Dice ↓
- **Dice / IoU bar chart** (mean ± std across folds, one bar per experiment)
- **Per-class Dice heatmap + bar chart** (meningioma / glioma / pituitary)
- **Per-fold scatter plot** — individual fold dots + mean line
- Saved CSVs + PNGs under `outputs/tables/segmentation/<dataset>/` and
  `outputs/figures/segmentation/<dataset>/comparison/`

**Prerequisites:** run `05_test.ipynb` for each experiment you want to compare.
This notebook reads from Drive directly (tables are small; no copy_to_local needed).

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

## Cell 3 — What to compare

Set `DATASET` and `SPLIT_SCHEME` to match the experiments you want to compare.

- `EXPERIMENTS = None` auto-discovers every experiment that has a `cv_summary.csv`.
- Override with a list of experiment names to restrict the comparison.

In [ ]:
# ── What to compare ──────────────────────────────────────────────────────────
DATASET      = "figshare"       # "figshare" | "brats2020"
SPLIT_SCHEME = "image_level"    # "image_level" | "patient_level"

# Experiments to include (None = auto-discover all with cv_summary.csv)
# Override example: ["03_dicebce_image_level", "07_unetpp_effb4_dicebce_image_level"]
EXPERIMENTS = None

print(f"dataset     : {DATASET}")
print(f"split_scheme: {SPLIT_SCHEME}")
print(f"experiments : {'auto-discover' if EXPERIMENTS is None else EXPERIMENTS}")

## Cell 4 — Load results from Drive

Reads `cv_summary.csv`, `cv_class_summary.csv`, and `cv_results.csv` for each
experiment. Experiments without results are listed but not included.

In [ ]:
import pandas as pd
from pathlib import Path

tables_root = DRIVE_ROOT / "outputs" / "tables" / "segmentation" / DATASET

# Discover experiment directories
if EXPERIMENTS is None:
    exp_names = sorted(p.parent.name for p in tables_root.glob("*/cv_summary.csv"))
else:
    exp_names = list(EXPERIMENTS)

print(f"Scanning {len(exp_names)} experiment(s) under {tables_root}")

summary_rows, class_rows, results_rows = [], [], []
found, missing = [], []

for exp_name in exp_names:
    exp_dir = tables_root / exp_name
    s_path  = exp_dir / "cv_summary.csv"
    c_path  = exp_dir / "cv_class_summary.csv"
    r_path  = exp_dir / "cv_results.csv"

    if not s_path.exists():
        missing.append(exp_name)
        continue

    s = pd.read_csv(s_path)
    if "split_scheme" in s.columns:
        s = s[s["split_scheme"] == SPLIT_SCHEME]
    if s.empty:
        print(f"  {exp_name}: no rows for split_scheme={SPLIT_SCHEME!r}")
        missing.append(exp_name)
        continue

    summary_rows.append(s)
    found.append(exp_name)

    if c_path.exists():
        class_rows.append(pd.read_csv(c_path))
    if r_path.exists():
        results_rows.append(pd.read_csv(r_path))

if missing:
    print(f"\nnot yet run (skipped): {missing}")
if not summary_rows:
    raise RuntimeError(
        f"No cv_summary.csv found under {tables_root} for split_scheme={SPLIT_SCHEME!r}.\n"
        f"Run 05_test.ipynb for at least one experiment first."
    )

comparison    = (
    pd.concat(summary_rows, ignore_index=True)
    .sort_values("dice_micro_mean", ascending=False)
    .reset_index(drop=True)
)
class_compare = pd.concat(class_rows,   ignore_index=True) if class_rows   else pd.DataFrame()
results_all   = pd.concat(results_rows,  ignore_index=True) if results_rows else pd.DataFrame()

print(f"\nloaded {len(found)} experiment(s) (ranked by Dice):")
for _, row in comparison.iterrows():
    print(f"  {row['experiment_name']:<45} Dice={row['dice_micro_mean']:.4f} \u00b1 {row['dice_micro_std']:.4f}")

## Cell 5 — Ranked comparison table

All experiments sorted by mean Dice descending. Green = Dice ≥ 0.85, yellow ≥ 0.80.

In [ ]:
from IPython.display import display

_cols = [c for c in [
    "experiment_name", "n_folds", "n_test_images_total",
    "dice_micro_mean", "dice_micro_std", "report_dice_micro",
    "iou_micro_mean",  "iou_micro_std",  "report_iou_micro",
    "sensitivity_micro_mean", "precision_micro_mean",
] if c in comparison.columns]

_fmt = {c: "{:.4f}" for c in _cols if c.endswith(("_mean", "_std"))}

def _color_dice(col):
    return col.map(
        lambda v: "background-color: #81c784" if v >= 0.85
        else "background-color: #c8e6c9" if v >= 0.80
        else "background-color: #fff9c4" if v >= 0.70
        else "background-color: #ffcdd2"
    )

display(
    comparison[_cols].style
    .format(_fmt, na_rep="\u2014")
    .apply(_color_dice, subset=["dice_micro_mean"])
    .set_caption(
        f"Cross-experiment comparison \u2014 {DATASET} / {SPLIT_SCHEME} (sorted by Dice \u2193)"
    )
)

best = comparison.iloc[0]
print(f"\nBest: {best['experiment_name']}  Dice={best['dice_micro_mean']:.4f} \u00b1 {best['dice_micro_std']:.4f}")
if len(comparison) > 1:
    worst = comparison.iloc[-1]
    delta = best["dice_micro_mean"] - worst["dice_micro_mean"]
    print(f"Span: {delta:.4f} Dice points across {len(comparison)} experiments")

## Cell 6 — Dice / IoU bar chart

Mean ± std across folds. Experiments sorted by Dice descending (same order as Cell 5).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

exp_names_  = comparison["experiment_name"].tolist()
dice_means  = comparison["dice_micro_mean"].tolist()
dice_stds   = comparison["dice_micro_std"].tolist()
iou_means   = comparison["iou_micro_mean"].tolist()
iou_stds    = comparison["iou_micro_std"].tolist()

# Short labels: strip the common split_scheme suffix
labels = [n.replace(f"_{SPLIT_SCHEME}", "") for n in exp_names_]

x = np.arange(len(labels))
w = 0.35

fig_bar, ax = plt.subplots(figsize=(max(8, len(labels) * 1.5), 5))
bars_d = ax.bar(x - w/2, dice_means, w, yerr=dice_stds, capsize=4,
                label="Dice (micro)", color="#1976D2", alpha=0.85, error_kw={"elinewidth": 1.2})
bars_i = ax.bar(x + w/2, iou_means,  w, yerr=iou_stds,  capsize=4,
                label="IoU (micro)",  color="#43A047", alpha=0.85, error_kw={"elinewidth": 1.2})

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_ylim(0.5, 1.0)
ax.set_title(f"Segmentation experiment comparison \u2014 {DATASET} / {SPLIT_SCHEME}")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
ax.axhline(y=0.80, color="grey", linestyle="--", alpha=0.4, linewidth=0.8)

for bar in bars_d:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.004,
            f"{h:.3f}", ha="center", va="bottom", fontsize=7.5)

fig_bar.tight_layout()
plt.show()
print("Bar chart rendered.")

## Cell 7 — Per-class Dice comparison

One column per experiment, one row per tumor class. Skipped if no `cv_class_summary.csv` was found.

In [ ]:
if class_compare.empty:
    print("No cv_class_summary.csv loaded — skipping per-class comparison.")
    fig_class = None
else:
    # Pivot: index=tumor_class, columns=experiment, values=dice_micro_mean
    pivot = class_compare.pivot_table(
        index="tumor_class",
        columns="experiment_name",
        values="dice_micro_mean",
    )
    pivot.columns = [c.replace(f"_{SPLIT_SCHEME}", "") for c in pivot.columns]

    # Sort columns by overall Dice rank (same order as Cell 5)
    rank_order = [n.replace(f"_{SPLIT_SCHEME}", "") for n in comparison["experiment_name"]]
    pivot = pivot[[c for c in rank_order if c in pivot.columns]]

    print("Per-class Dice mean (rows=class, cols=experiment, sorted by overall Dice \u2193):")
    display(pivot.style.format("{:.4f}", na_rep="\u2014")
            .background_gradient(cmap="RdYlGn", vmin=0.6, vmax=1.0)
            .set_caption(f"Per-class Dice \u2014 {DATASET} / {SPLIT_SCHEME}"))

    fig_class, ax2 = plt.subplots(figsize=(max(8, len(pivot.columns) * 1.5), 4))
    pivot.T.plot(kind="bar", ax=ax2, alpha=0.85)
    ax2.set_xlabel("")
    ax2.set_ylabel("Dice (micro mean)")
    ax2.set_title(f"Per-class Dice \u2014 {DATASET} / {SPLIT_SCHEME}")
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=30, ha="right", fontsize=9)
    ax2.grid(axis="y", alpha=0.3)
    ax2.legend(title="class", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    ax2.set_ylim(0.5, 1.0)
    fig_class.tight_layout()
    plt.show()
    print("Per-class chart rendered.")

## Cell 8 — Per-fold scatter

Each dot is one fold's Dice. The red line is the cross-fold mean.
Shows variance across folds — a wide spread indicates an unstable experiment.

In [ ]:
if results_all.empty:
    print("No cv_results.csv loaded — skipping per-fold scatter.")
    fig_scatter = None
else:
    results_all["exp_label"] = results_all["experiment_name"].str.replace(
        f"_{SPLIT_SCHEME}", "", regex=False)

    # Keep same left-to-right rank order as Cell 5
    rank_order = [n.replace(f"_{SPLIT_SCHEME}", "") for n in comparison["experiment_name"]]
    exp_labels = [e for e in rank_order if e in results_all["exp_label"].unique()]
    exp_to_x   = {e: i for i, e in enumerate(exp_labels)}

    fig_scatter, ax3 = plt.subplots(figsize=(max(8, len(exp_labels) * 1.5), 4))

    for _, row in results_all.iterrows():
        lbl = row["exp_label"]
        if lbl not in exp_to_x:
            continue
        xi = exp_to_x[lbl]
        ax3.scatter(xi, row["dice_micro"], color="#1976D2", alpha=0.6, s=45, zorder=3)

    means = results_all[results_all["exp_label"].isin(exp_labels)].groupby("exp_label")["dice_micro"].mean()
    for exp, m in means.items():
        xi = exp_to_x[exp]
        ax3.hlines(m, xi - 0.3, xi + 0.3, colors="#D32F2F", linewidth=2.2, zorder=4)

    ax3.set_xticks(list(exp_to_x.values()))
    ax3.set_xticklabels(exp_labels, rotation=30, ha="right", fontsize=9)
    ax3.set_ylabel("Dice (micro)")
    ax3.set_title(f"Per-fold Dice — {DATASET} / {SPLIT_SCHEME}  (— = mean)")
    ax3.set_ylim(0.5, 1.0)
    ax3.grid(axis="y", alpha=0.3)
    fig_scatter.tight_layout()
    plt.show()
    print("Per-fold scatter rendered.")

## Cell 9 — Save outputs to Drive

Writes CSVs and PNGs directly to Drive (tables are small; no local-SSD sync needed).

Output paths:
- `outputs/tables/segmentation/<dataset>/comparison_cv_summary_<scheme>.csv`
- `outputs/tables/segmentation/<dataset>/comparison_cv_class_summary_<scheme>.csv`
- `outputs/figures/segmentation/<dataset>/comparison/dice_bar_<scheme>.png`
- `outputs/figures/segmentation/<dataset>/comparison/class_dice_bar_<scheme>.png`
- `outputs/figures/segmentation/<dataset>/comparison/fold_scatter_<scheme>.png`

In [ ]:
out_tables  = DRIVE_ROOT / "outputs" / "tables"  / "segmentation" / DATASET
out_figures = DRIVE_ROOT / "outputs" / "figures" / "segmentation" / DATASET / "comparison"
out_tables.mkdir (parents=True, exist_ok=True)
out_figures.mkdir(parents=True, exist_ok=True)

sfx = SPLIT_SCHEME

# ── Tables ───────────────────────────────────────────────────────────────
p = out_tables / f"comparison_cv_summary_{sfx}.csv"
comparison.to_csv(p, index=False)
print(f"saved {p.name}")

if not class_compare.empty:
    p2 = out_tables / f"comparison_cv_class_summary_{sfx}.csv"
    class_compare.to_csv(p2, index=False)
    print(f"saved {p2.name}")

# ── Figures ──────────────────────────────────────────────────────────────
if fig_bar is not None:
    fig_bar.savefig(out_figures / f"dice_bar_{sfx}.png", dpi=150, bbox_inches="tight")
    print(f"saved dice_bar_{sfx}.png")

if fig_class is not None:
    fig_class.savefig(out_figures / f"class_dice_bar_{sfx}.png", dpi=150, bbox_inches="tight")
    print(f"saved class_dice_bar_{sfx}.png")

if fig_scatter is not None:
    fig_scatter.savefig(out_figures / f"fold_scatter_{sfx}.png", dpi=150, bbox_inches="tight")
    print(f"saved fold_scatter_{sfx}.png")

print(f"\ntables  -> {out_tables.relative_to(DRIVE_ROOT)}")
print(f"figures -> {out_figures.relative_to(DRIVE_ROOT)}")